# Regular Expressions



```{contents}
:local:
:depth: 2
```


```{index} Read a Pattern as a Description of Accepted Text
```



```{index} Read a Pattern as a Description of Accepted Text
```

## Read a Pattern as a Description of Accepted Text



A product label consists of two uppercase ASCII letters, a hyphen and four digits. Before choosing regular-expression symbols, write accepted and rejected examples. AB-0042 fits; A-0042 has too few letters; AB-42 has too few digits; and a larger sentence containing AB-0042 is not itself one complete label. This last distinction separates finding a matching substring from validating an entire field.

For this deliberately narrow label, the pattern `\A[A-Z]{2}-[0-9]{4}\z` uses beginning/end-of-input anchors, a range of letters repeated twice, a literal hyphen and four ASCII digits. It states shape, not whether AB is a known department. In a C# verbatim string literal the pattern can be written `@"\A[A-Z]{2}-[0-9]{4}\z"`; ordinary C# string escaping and regex escaping are different layers. The @ belongs to the string literal form, not to the regex language.

| Pattern part | Meaning under this contract |
| --- | --- |
| \A | Beginning of the input |
| [A-Z]{2} | Exactly two letters from the stated ASCII range |
| - | The separator |
| [0-9]{4} | Exactly four ASCII digits |
| \z | End of the input |

A group such as `(?<number>[0-9]{4})` names a captured part. Extracting number as text retains leading zeros; converting it to an integer would answer a different question. Check whether a match succeeded before treating group contents as a valid field. A match object exists even when the match is unsuccessful, so object existence alone is not the success test.

Regex replacement also has a policy. Removing every nondigit character may produce a digit-only result while silently discarding an unexpected department or separator. Decide whether malformed input should be repaired, rejected or retained with a warning. A cleaning rule is not equivalent to a validation rule.

Guided exercise: test AB-0042, AB-0042 followed by a newline, lowercase ab-0042, a matching code embedded in a sentence and an empty string. Predict results under the exact anchored policy above. Then change the requirement to finding labels inside messages and explain which anchors must change. Do not broaden the policy merely because one difficult input fails. For untrusted complex patterns and long inputs, consider bounded matching behavior; the core fixed-shape examples do not teach arbitrary patterns as safe substitutes for a parser.

For related explanation, see the further reading below.[^development-reading]


In [1]:
using System;
using System.Text.RegularExpressions;
string pattern = @"\A[A-Z]{2}-[0-9]{4}\z";
Console.WriteLine(Regex.IsMatch("AB-0042", pattern));
Console.WriteLine(Regex.IsMatch("AB-0042\n", pattern));
Console.WriteLine(Regex.IsMatch("Message AB-0042", pattern));


True
False
False


Regular expressions (regex) are patterns that describe text. They extend the string search and replace operations from the previous section, letting you match flexible, structured patterns instead of exact strings.

> Regex lives in the `System.Text.RegularExpressions` namespace.

In [ ]:
using System.Text.RegularExpressions;

```{index} Checking for a Match
```



```{index} Checking for a Match
```

## Checking for a Match




`Regex.IsMatch` returns `true` if the pattern appears anywhere in the string:

In [ ]:
using System.Text.RegularExpressions;

// Does the string contain any digit?
bool hasDigit = Regex.IsMatch("Score: 98", @"\d");
Console.WriteLine(hasDigit);  // True

// Does it start with a capital letter?
bool startsUpper = Regex.IsMatch("Alice", @"^[A-Z]");
Console.WriteLine(startsUpper);  // True

```{index} Common Pattern Elements
```



```{index} Common Pattern Elements
```

## Common Pattern Elements




| Pattern | Meaning | Example match |
|---------|---------|---------------|
| `\d` | any digit | `7`, `0` |
| `\w` | word character (letter, digit, `_`) | `a`, `Z`, `3` |
| `\s` | whitespace | space, tab |
| `.` | any character except newline | `x`, `!` |
| `^` | start of string | `^Hello` matches `Hello world` |
| `$` | end of string | `\.txt$` matches `file.txt` |
| `[abc]` | one of a, b, c | `[aeiou]` matches any vowel |
| `[A-Z]` | range | any uppercase letter |
| `+` | one or more | `\d+` matches `42` |
| `*` | zero or more | `\d*` matches `` or `99` |
| `?` | zero or one | `colou?r` matches `color` or `colour` |

```{index} Extracting a Match
```



```{index} Extracting a Match
```

## Extracting a Match




`Regex.Match` returns the first match object:

General extraction forms (patterns):

```csharp
Match result = Regex.Match(input, pattern);
if (result.Success)
    Console.WriteLine(result.Value);
MatchCollection results = Regex.Matches(input, pattern);
string revised = Regex.Replace(input, pattern, replacement);
```

`Match` is an object describing one match, not the matched text itself. `Success` tells whether a match exists; `Value` is its text, `Index` is its zero-based starting position, and `Length` is the number of characters matched. `Matches` returns a collection, which can be empty. `Replace` returns a new string; it does not edit the original. These are the same call/property/collection distinctions used with strings and other collection families.


In [ ]:
using System.Text.RegularExpressions;

string line = "Student: Alice, Score: 98";
Match m = Regex.Match(line, @"\d+");

if (m.Success)
    Console.WriteLine($"First number found: {m.Value}");  // 98

```{index} Finding All Matches
```



```{index} Finding All Matches
```

## Finding All Matches




`Regex.Matches` returns every match:

In [ ]:
using System.Text.RegularExpressions;

string data = "Scores: 85, 92, 78, 90";
MatchCollection matches = Regex.Matches(data, @"\d+");

foreach (Match m in matches)
    Console.WriteLine(m.Value);
// 85  92  78  90

```{index} Replacing with a Pattern
```



```{index} Replacing with a Pattern
```

## Replacing with a Pattern




`Regex.Replace` substitutes every match:

In [ ]:
using System.Text.RegularExpressions;

// Normalize multiple spaces to one
string messy = "Alice,  98,   CS";
string clean = Regex.Replace(messy, @"\s{2,}", " ");
Console.WriteLine(clean);  // Alice,  98,   CS -> "Alice, 98, CS"

In [ ]:
using System.Text.RegularExpressions;

// Remove all non-digit characters
string phone = "(312) 555-0190";
string digits = Regex.Replace(phone, @"[^\d]", "");
Console.WriteLine(digits);  // 3125550190

```{index} Using Groups to Extract Fields
```



```{index} Using Groups to Extract Fields
```

## Using Groups to Extract Fields




Parentheses create **capture groups** — named parts of a match:

In [ ]:
using System.Text.RegularExpressions;

string record = "Alice,98,CS";
Match m = Regex.Match(record, @"^(?<name>\w+),(?<score>\d+),(?<major>\w+)$");

if (m.Success)
{
    Console.WriteLine($"Name:  {m.Groups["name"].Value}");
    Console.WriteLine($"Score: {m.Groups["score"].Value}");
    Console.WriteLine($"Major: {m.Groups["major"].Value}");
}

```{index} Validation Patterns
```



```{index} Validation Patterns
```

## Validation Patterns




Common tasks well-suited to regex:

In [ ]:
using System.Text.RegularExpressions;

// Simple email check (not exhaustive)
bool isEmail = Regex.IsMatch("user@example.com", @"^[\w.+-]+@[\w-]+\.[a-z]{2,}$");
Console.WriteLine($"valid email: {isEmail}");

// Zip code (5 digits)
bool isZip = Regex.IsMatch("60660", @"^\d{5}$");
Console.WriteLine($"valid zip: {isZip}");

// Date format YYYY-MM-DD
bool isDate = Regex.IsMatch("2026-03-17", @"^\d{4}-\d{2}-\d{2}$");
Console.WriteLine($"valid date: {isDate}");

```{index} When to Use Regex vs. String Methods
```



```{index} When to Use Regex vs. String Methods
```

## When to Use Regex vs. String Methods




| Use `string` methods | Use regex |
|---------------------|-----------|
| Exact match or replace | Pattern-based match |
| Fixed delimiter `Split(',')` | Variable whitespace or delimiters |
| `Contains`, `StartsWith` | Character classes, repetition |
| Performance-critical loops | One-time validation or extraction |

Regex adds power but also complexity — prefer `Split`, `Trim`, and `Contains` when the pattern is simple and fixed.

```{index} regular expressions; named groups; failed match
```

### Trace Named Fields and a Failed Match


A delivery record has exactly `SKU=`, two ASCII letters, a hyphen, and four digits. Capturing a named field lets later code use the field's meaning instead of its position in a substring. The syntax `(?<name>pattern)` labels a capture; `result.Groups["name"].Value` reads its text **after a successful match**. The anchors `\A` and `\z` require the entire input to obey the contract.

| Input | Match? | Captured `sku` | Reason |
| --- | --- | --- | --- |
| `SKU=AB-0042` | yes | `AB-0042` | exact shape |
| `SKU=AB-42` | no | do not use | only two digits |
| `note SKU=AB-0042` | no | do not use | extra prefix |

A capture is still a string, even when it contains digits. Valid shape also does not establish that the SKU exists in an inventory.


In [1]:
using System;
using System.Text.RegularExpressions;
string pattern = @"\ASKU=(?<sku>[A-Z]{2}-[0-9]{4})\z";
foreach (string input in new[] { "SKU=AB-0042", "SKU=AB-42", "note SKU=AB-0042" })
{
    Match result = Regex.Match(input, pattern);
    Console.WriteLine(result.Success ? $"sku={result.Groups["sku"].Value}" : "rejected");
}
string original = "dock    A";
string normalized = Regex.Replace(original, @" +", " ");
Console.WriteLine($"original=[{original}], revised=[{normalized}]");


sku=AB-0042
rejected
rejected
original=[dock    A], revised=[dock A]


The first line reports `sku=AB-0042`; the next two report `rejected`. Reading `Groups` without checking `Success` can silently yield empty text rather than a meaningful field. The last line preserves the original four spaces and shows one space in the revised value. Here the replacement pattern uses a literal space plus `+`, so it changes runs of ordinary spaces; `\s+` would additionally match other whitespace, such as tabs and line breaks. Choose the narrower contract when those characters must remain distinct.

Guided variation: accept `Ticket=4821` as a whole record and extract only the digits into a group called `id`. Reject `Ticket=48x1` and `prefix Ticket=4821`. The starter's unanchored match currently accepts the prefixed case; repair both the pattern and the group lookup.


In [1]:
using System;
using System.Text.RegularExpressions;
string pattern = @"Ticket=(?<wrong>[0-9]{4})"; // TODO: whole input, group named id.
foreach (string input in new[] { "Ticket=4821", "Ticket=48x1", "prefix Ticket=4821" })
{
    Match m = Regex.Match(input, pattern);
    Console.WriteLine(m.Success ? m.Groups["wrong"].Value : "rejected");
}


4821
rejected
4821


In [1]:
using System;
using System.Text.RegularExpressions;
// Solution
string pattern = @"\ATicket=(?<id>[0-9]{4})\z";
foreach (string input in new[] { "Ticket=4821", "Ticket=48x1", "prefix Ticket=4821" })
{
    Match m = Regex.Match(input, pattern);
    Console.WriteLine(m.Success ? m.Groups["id"].Value : "rejected");
}


4821
rejected
rejected


The solution prints `4821`, `rejected`, `rejected`. An unanchored expression asks whether a matching substring exists; adding whole-input anchors changes the question to record validation. For another variation, collect all ticket numbers in a message with `Matches` instead; in that task, rejecting surrounding prose would be the wrong contract. Use these contrasts before the independent patterns below.


```{index} Practice
```

## Practice




1. Write a pattern to match a C# `int` or `double` literal (e.g. `42`, `-3`, `3.14`).
2. Extract all words (sequences of `\w+`) from a sentence and print each on its own line.
3. Validate that a filename ends with `.cs` or `.txt`.
4. Replace all tab characters in a file line with four spaces.

```{rubric} Footnotes
```


[^development-reading]: Further reading: [Microsoft, Anchors in .NET Regular Expressions](https://learn.microsoft.com/en-us/dotnet/standard/base-types/anchors-in-regular-expressions). The worked business scenarios and guided reasoning here are original teaching examples.
